# Ignite-3B Session S01 - Baseline Cond A + weco Cond B kick

**Goal**: eval v_0 (Qwen2.5-3B base) on Ignite benches + start weco Cond B run.

Setup:
1. Accelerator = GPU T4 x2
2. Internet ON
3. Persistence = Variables and Files
4. Kaggle Secret HF_TOKEN (write) + ANTHROPIC_API_KEY (Cond B)

Runs Cond A eval, uploads results dataset, kicks weco run in bg.

In [ ]:
BASE_MODEL = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
BENCHES_A = ['omni_math', 'livecodebench', 'matharena', 'aime']
N_OMNI = 100   # cap eval to fit 8-10h Kaggle
N_LCB = 50
N_MATHARENA = 50
N_AIME = 30
print(f'base={BASE_MODEL}, benches={BENCHES_A}')

In [ ]:
!pip install -q -U 'transformers>=4.46.0' 'peft>=0.13.0' 'datasets>=3.0.0' 'accelerate>=1.0.0' 'unsloth>=2025.1.0' 'trl>=0.12.0' 'vllm>=0.6.0' 'math-verify>=0.5.2' 'latex2sympy2' 'sympy' 'weco>=0.3.40' 'scipy' kaggle huggingface_hub

In [ ]:
import os, subprocess
if not os.path.exists('/kaggle/working/caracal-1'):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 's07-hybrid-agentic',
                    'https://github.com/iterate-labs-ai/caracal-1.git',
                    '/kaggle/working/caracal-1'], check=True)
os.chdir('/kaggle/working/caracal-1')
rev = subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip()
print(f'HEAD={rev}')

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    from huggingface_hub import login
    login(token=hf_token)
    print('HF login OK')
except Exception as e:
    print(f'HF login skipped ({e})')

In [ ]:
import subprocess
for script in ['build_omni_math', 'build_livecodebench', 'build_matharena', 'build_aime']:
    subprocess.run(['python', '-m', f'data.ignite.{script}'], check=False)
subprocess.run(['ls', '-la', 'data/ignite/'], check=False)

In [ ]:
import subprocess, sys
os.makedirs('results', exist_ok=True)
subprocess.run([sys.executable, '-m', 'eval.ignite.run_all_ignite',
                '--base', BASE_MODEL,
                '--benches', *BENCHES_A,
                '--n-omni-math', str(N_OMNI),
                '--n-livecodebench', str(N_LCB),
                '--n-matharena', str(N_MATHARENA),
                '--n-aime', str(N_AIME),
                '--out', 'results/cond_A.json'], check=True)

In [ ]:
import json
res = json.load(open('results/cond_A.json'))
for k, v in res.items():
    if isinstance(v, dict) and 'accuracy' in v:
        print(f"{k:20s}: {v['accuracy']*100:5.1f}%  n={v.get('n', 0)}")

In [ ]:
import json
print('--- Cond A baseline results ---')
res = json.load(open('results/cond_A.json'))
for k, v in res.items():
    if isinstance(v, dict) and 'accuracy' in v:
        print(f"{k:20s}: {v['accuracy']*100:5.1f}%  n={v.get('n', 0)}  CI[{v.get('ci_95_low',0)*100:.1f}-{v.get('ci_95_high',0)*100:.1f}]")
print('\nJSON saved to /kaggle/working/caracal-1/results/cond_A.json')